# Attention Mechanism from Scratch
## Self-Attention | Multi-Head Attention | Mini Transformer

The Transformer architecture ("Attention Is All You Need") revolutionized NLP and now powers GPT, BERT, and all modern LLMs.

---

In [ ]:
# CELL 1: Setup
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt, math
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42; torch.manual_seed(SEED)
OUTPUT_DIR = Path('outputs'); OUTPUT_DIR.mkdir(exist_ok=True)
print(f'Device: {device}')

In [ ]:
# CELL 2: Scaled Dot-Product Attention (Core Building Block)
def scaled_dot_product_attention(query, key, value, mask=None):
    """
    Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V
    
    Args:
        query: (batch, heads, seq_len, d_k)
        key:   (batch, heads, seq_len, d_k)
        value: (batch, heads, seq_len, d_v)
        mask:  optional mask for causal/padding attention
    Returns:
        output: (batch, heads, seq_len, d_v)
        attention_weights: (batch, heads, seq_len, seq_len)
    """
    d_k = query.size(-1)
    # QK^T / sqrt(d_k) -- scaling prevents softmax saturation
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, value)
    return output, attention_weights

# Demo
batch, seq_len, d_model = 2, 8, 64
q = k = v = torch.randn(batch, 1, seq_len, d_model)
out, attn = scaled_dot_product_attention(q, k, v)
print(f"Input shape: {q.shape}")
print(f"Output shape: {out.shape}")
print(f"Attention weights shape: {attn.shape}")

In [ ]:
# CELL 3: Multi-Head Attention
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention allows the model to attend to information
    from different representation subspaces at different positions.
    
    Instead of one attention function with d_model dims, we project
    Q, K, V into h heads of d_k dims each, run attention in parallel,
    then concatenate and project back.
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)  # Query projection
        self.W_k = nn.Linear(d_model, d_model)  # Key projection
        self.W_v = nn.Linear(d_model, d_model)  # Value projection
        self.W_o = nn.Linear(d_model, d_model)  # Output projection
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        
        # Linear projections and reshape to (batch, heads, seq, d_k)
        Q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # Apply attention
        attn_out, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and project
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        output = self.W_o(attn_out)
        return output, attn_weights

mha = MultiHeadAttention(d_model=64, num_heads=8)
x = torch.randn(2, 10, 64)  # (batch=2, seq=10, dim=64)
out, weights = mha(x, x, x)
print(f"MHA output: {out.shape} | Attention: {weights.shape}")

In [ ]:
# CELL 4: Positional Encoding
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (from 'Attention Is All You Need')."""
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

# Visualize
pe = PositionalEncoding(64)
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(pe.pe[0, :50, :].numpy(), aspect='auto', cmap='RdBu')
ax.set_xlabel('Dimension'); ax.set_ylabel('Position')
ax.set_title('Sinusoidal Positional Encoding'); plt.colorbar(im)
plt.savefig(OUTPUT_DIR / 'positional_encoding.png', dpi=150); plt.show()

In [ ]:
# CELL 5: Mini Transformer Encoder Block
class TransformerBlock(nn.Module):
    """Single transformer encoder block: MHA + FFN + LayerNorm + Residual."""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout)
        )
    
    def forward(self, x, mask=None):
        # Self-attention with residual connection
        attn_out, attn_w = self.attention(x, x, x, mask)
        x = self.norm1(x + attn_out)
        # Feed-forward with residual connection
        x = self.norm2(x + self.ffn(x))
        return x, attn_w

class MiniTransformer(nn.Module):
    """Mini Transformer for sequence classification."""
    def __init__(self, vocab_size, d_model=128, num_heads=4, d_ff=256, n_layers=3, n_classes=2, max_len=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, num_heads, d_ff) for _ in range(n_layers)])
        self.classifier = nn.Linear(d_model, n_classes)
    
    def forward(self, x, mask=None):
        x = self.pos_enc(self.embedding(x))
        attn_maps = []
        for block in self.blocks:
            x, attn = block(x, mask)
            attn_maps.append(attn)
        # Use [CLS]-like pooling (mean of sequence)
        pooled = x.mean(dim=1)
        return self.classifier(pooled), attn_maps

model = MiniTransformer(vocab_size=10000, n_classes=2).to(device)
dummy = torch.randint(0, 10000, (4, 32)).to(device)
logits, attn = model(dummy)
print(f"Logits: {logits.shape} | Attention maps: {len(attn)} layers")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")
torch.save(model.state_dict(), OUTPUT_DIR / 'mini_transformer.pt')
print("Attention mechanism notebook complete!")